# TrialMatch: base validation, full LoRA training and comparison

Use **Qwen2.5-1.5B-Instruct**, starting from the original pinned base checkpoint, on your
Colab A100. The prior one-step smoke adapter is not used. This notebook evaluates the
base on validation, trains for three epochs, selects a fine-tuned checkpoint using validation
macro-F1, and compares the reloaded selected adapter against the same base examples.

**The test set stays unused.** These are development results. This notebook does not yet
implement the assignment's custom KV, tiled-attention or speculative decoders.

Choose **Runtime → Change runtime type → A100 GPU**. Use the **updated** companion
`trialmatch-colab-starter.zip`. An older setup ZIP does not contain the new training scripts.


## 1. Upload the updated source bundle

Upload the companion ZIP when prompted, including if you used notebook 01 in this session.
Each bundle version gets its own project directory so an older upload cannot silently win.


In [ ]:
from pathlib import Path
import hashlib
import os
import stat
import zipfile
from google.colab import files

uploaded = files.upload()
expected = 'trialmatch-colab-starter.zip'
if expected not in uploaded:
    raise ValueError(f'Upload the updated {expected}.')
bundle_bytes = uploaded[expected]
BUNDLE_SHA256 = hashlib.sha256(bundle_bytes).hexdigest()
PROJECT = Path('/content/trialmatch-training-' + BUNDLE_SHA256[:12])
archive_path = Path('/content') / expected
archive_path.write_bytes(bundle_bytes)
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive_path) as archive:
    if 'scripts/run_training.py' not in archive.namelist():
        raise ValueError('This is an older bundle. Upload the updated training bundle.')
    for entry in archive.infolist():
        if not (PROJECT / entry.filename).resolve().is_relative_to(PROJECT.resolve()):
            raise ValueError('Archive contains an unsafe path')
        if stat.S_ISLNK(entry.external_attr >> 16):
            raise ValueError('Archive contains a symbolic link')
    archive.extractall(PROJECT)
del uploaded, bundle_bytes
os.chdir(PROJECT)
os.environ['HF_HOME'] = '/content/trialmatch-hf-cache'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['MPLCONFIGDIR'] = '/content/trialmatch-matplotlib'
print('Project:', PROJECT)
print('Bundle SHA256:', BUNDLE_SHA256)


## 2. Install the training stack and check CUDA

Preserve Colab's preinstalled CUDA PyTorch. Actual versions are saved with the run. If a
restart is requested, restart the runtime, upload the same bundle and rerun cells in order.


In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-colab.txt', '-e', '.'], check=True)
subprocess.run([sys.executable, '-c',
    'import torch, transformers, peft; '
    'assert torch.cuda.is_available(), "Select a GPU runtime"; '
    'print(torch.cuda.get_device_name(0)); '
    'print("torch", torch.__version__, "transformers", transformers.__version__, "peft", peft.__version__)'], check=True)


## 3. Mount Drive for checkpoints and logs

Resume checkpoints include optimizer state and must survive a runtime disconnect.
Drive is recommended. With Drive disabled, download results before disconnecting;
the portable export contains the selected adapter and logs but excludes optimizer state.


In [ ]:
USE_GOOGLE_DRIVE = True
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT = Path('/content/drive/MyDrive/TrialMatch/results')
else:
    OUTPUT_ROOT = PROJECT / 'results/colab'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


## 4. Recreate the fixed split and run the checks

This is the same immutable source and 37/8/8 patient split used in the successful setup.
Dataset preparation creates all split files mechanically; training/evaluation opens only
train and validation. Do not inspect or tune on the test file.


In [ ]:
import json
manifest = json.loads((PROJECT / 'data/manifests/split_manifest.json').read_text())
subprocess.run([sys.executable, 'scripts/prepare_data.py', '--revision', manifest['dataset_revision']], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)


## 5. Choose the run and training settings

The initial settings use micro-batch 2 and accumulation 4: nominal effective batch 8.
The final short batch is retained and normalized by its actual supervised-token count.
These settings are not claimed to be optimal. Keep them unchanged when resuming.

For a new run leave `RESUME_RUN_PATH` empty. To resume after disconnecting, upload the
same source bundle, run cells 1–4, and paste the earlier **full run directory** into that field.
Resume restores the last completed epoch. An interrupted partial epoch is repeated;
mid-epoch optimizer progress is not restored. Changed code, data, settings or recorded
runtime environment cause resume to fail rather than silently mix experiments.


In [ ]:
from datetime import datetime, timezone
import re

RESUME_RUN_PATH = ''  # Example: /content/drive/MyDrive/TrialMatch/results/training-...
EPOCHS = 3
TRAIN_BATCH_SIZE = 2
ACCUMULATION_STEPS = 4
EVAL_BATCH_SIZE = 4
LEARNING_RATE = 0.0002

settings = (PROJECT / 'configs/experiment.toml').read_text()
for key, value in {'epochs': EPOCHS, 'per_device_batch_size': TRAIN_BATCH_SIZE,
                   'gradient_accumulation_steps': ACCUMULATION_STEPS,
                   'batch_size': EVAL_BATCH_SIZE, 'learning_rate': LEARNING_RATE}.items():
    settings, matches = re.subn(r'(?m)^' + re.escape(key) + r'\s*=.*$', f'{key} = {value}', settings)
    if matches != 1:
        raise ValueError(f'Expected one configuration entry for {key}')
RUN_CONFIG = PROJECT / 'configs/colab_run.toml'
RUN_CONFIG.write_text(settings)
if RESUME_RUN_PATH:
    RUN_DIR = Path(RESUME_RUN_PATH)
    if not (RUN_DIR / 'run.json').exists():
        raise ValueError('Resume path must point to an existing training run with run.json')
else:
    RUN_DIR = OUTPUT_ROOT / ('training-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    RUN_DIR.mkdir(parents=True, exist_ok=False)
print('Keep this path for resuming:', RUN_DIR)
print('Training micro-batch / accumulation:', TRAIN_BATCH_SIZE, '/', ACCUMULATION_STEPS)


## 6. Run baseline evaluation and full training

The script saves:
- base predictions and metrics for every validation example;
- training loss and learning-rate logs, plus full checkpoints at completed epochs;
- generated validation metrics after each epoch;
- the selected adapter pointer and a fresh-reload validation consistency check;
- before/after tables, per-class/type metrics, evidence scores and training curves.

Selection uses validation classification macro-F1, then evidence micro-F1; exact ties keep
the earlier epoch. Invalid outputs remain in evaluation as failures. This job is longer than
the one-step test. Observe progress in the cell and persistent `console.log`.


In [ ]:
command = [sys.executable, '-u', 'scripts/run_training.py',
           '--output-dir', str(RUN_DIR), '--config', str(RUN_CONFIG)]
if RESUME_RUN_PATH:
    command.append('--resume')
with (RUN_DIR / 'console.log').open('a') as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
        log.write(line)
        log.flush()
    return_code = process.wait()
if return_code:
    raise RuntimeError(f'Run failed. Preserve {RUN_DIR}/console.log, status.json and failure.txt if present.')


## 7. Inspect the measured comparison

Look at rare-class performance, evidence and invalid outputs, not accuracy alone.
The selected fine-tuned model may perform worse than the base; report that honestly.
The complete confusion matrices and fixed-label macro conventions are in comparison.json.


In [ ]:
from IPython.display import Markdown, display, Image
print((RUN_DIR / 'status.json').read_text())
display(Markdown((RUN_DIR / 'COMPARISON.md').read_text()))
display(Image(filename=str(RUN_DIR / 'training-validation-curves.png')))
print((RUN_DIR / 'checkpoint_reload.json').read_text())
print('Selected adapter:', (RUN_DIR / 'selected_adapter.json').read_text())


## 8. Download the portable results

The export includes logs, predictions, charts and only the selected adapter. Full optimizer
resume states and other adapters stay in the original Drive run directory. Keep that
folder until the project is complete; this download is not a resume archive.
Share the result ZIP for review before changing prompts or hyperparameters.


In [ ]:
result_zip = Path('/content') / (RUN_DIR.name + '-results.zip')
subprocess.run([sys.executable, 'scripts/export_training_results.py', '--run-dir', str(RUN_DIR),
                '--output', str(result_zip)], check=True)
files.download(str(result_zip))


## Next steps

Review validation errors and minority-label failures, then implement the required custom
inference mechanisms and browser application. Keep the fixed test set unused until all
choices are frozen. Educational screening prototype; not a clinical enrollment decision.
